# Playwright with JavaScript — Day 1
## 5 November 2026: Locators, Selectors and Assertions


**Prerequisite:** JavaScript fundamentals completed.

## Learning objectives
By the end of this class, students should be able to:
1. Understand how Playwright identifies web elements.
2. Use different locator strategies.
3. Handle multiple matching elements.
4. Interact with web elements.
5. Verify application behavior using assertions.
6. Write a complete Playwright test.


## 1. Understanding locators 

A **locator** is how Playwright identifies an element on a webpage: a textbox, button, link, checkbox, and so on.

Example HTML:

```html
<input type="text" id="username">
<button id="login">Login</button>
```

```javascript
page.locator('#username');
page.locator('#login');
```

### CSS selector mapping

| HTML attribute | CSS locator |
|---|---|
| `id="username"` | `#username` |
| `class="login-btn"` | `.login-btn` |
| `name="email"` | `[name="email"]` |
| `type="text"` | `[type="text"]` |

### Locator types

**A. `getByRole()`** — identifies elements by accessibility role and accessible name.

```javascript
page.getByRole('button', { name: 'Login' });
page.getByRole('link', { name: 'Home' });
page.getByRole('textbox', { name: 'Username' });
page.getByRole('checkbox', { name: 'Remember me' });
```

Common roles: `button`, `textbox`, `link`, `checkbox`, `radio`, `heading`, `combobox`, `img`.

**B. `getByText()`**

```javascript
page.getByText('Welcome');
page.getByText('Login successful');
```

**C. `getByLabel()`**

```javascript
page.getByLabel('Email');
page.getByLabel('Password');
```

**D. `getByPlaceholder()`**

```javascript
page.getByPlaceholder('Enter your email');
page.getByPlaceholder('Search here');
```

**E. `getByAltText()`**

```javascript
page.getByAltText('Company Logo');
```

**F. `getByTestId()`**

HTML: `<button data-testid="submit-btn">Submit</button>`

```javascript
page.getByTestId('submit-btn');
```

**G. `locator()` — CSS and XPath**

```javascript
page.locator('#username');
page.locator('.login-button');
page.locator('input[name="email"]');
page.locator('//button[text()="Login"]');
```

Prefer user-facing locators such as `getByRole()` when they reliably identify the intended element. CSS and XPath remain useful when needed.


## 2. Locator actions 
| Method | Purpose |
|---|---|
| `click()` | Click an element |
| `dblclick()` | Double-click |
| `fill()` | Enter or replace input text |
| `clear()` | Clear an input |
| `type()` | Type character by character |
| `press()` | Press a keyboard key |
| `check()` | Select checkbox/radio |
| `uncheck()` | Deselect checkbox |
| `hover()` | Hover over element |
| `focus()` | Focus an element |

```javascript
await page.getByPlaceholder('Enter Name').fill('Arun');
await page.getByRole('button', { name: 'Submit' }).click();
await page.getByLabel('Remember me').check();
await page.getByLabel('Remember me').uncheck();
await page.getByPlaceholder('Search').press('Enter');
```

### `fill()` versus `type()`

```javascript
await page.locator('#username').fill('Arun');
await page.locator('#username').type('Arun');
```

- `fill()` replaces the existing input value; prefer it for normal form input.
- `type()` simulates typing characters.
- `pressSequentially()` is also available when character-by-character keyboard input is specifically needed.


## 3. Handling multiple elements 

A page may contain several elements matching the same locator.

```javascript
const links = page.getByRole('link');
console.log(await links.count());
```

| Method | Purpose |
|---|---|
| `first()` | First match |
| `last()` | Last match |
| `nth(2)` | Third match |
| `count()` | Number of matches |

```javascript
await links.first().click();
await links.last().click();
await links.nth(2).click();
```

Indexing starts at zero: `0` = first, `1` = second, `2` = third.

```javascript
const buttons = page.getByRole('button');
const totalButtons = await buttons.count();
console.log('Total buttons:', totalButtons);
```

### Filtering

```javascript
const product = page.locator('.product')
    .filter({ hasText: 'Laptop' });
```

Actions such as `click()` generally require one matching element. Make the locator more specific or use `first()`, `last()` or `nth()` when appropriate.


## 4. Assertions in Playwright 

An **assertion** verifies whether the actual result matches the expected result. Actions perform operations; assertions verify results.

```javascript
const { test, expect } = require('@playwright/test');
```

### Page assertions

```javascript
await expect(page).toHaveTitle(/Automation Testing Practice/);
await expect(page).toHaveURL(/dashboard/);
```

### Locator assertions

```javascript
await expect(page.getByText('Welcome')).toBeVisible();
await expect(page.getByText('Loading')).toBeHidden();
await expect(page.locator('#username')).toBeAttached();
await expect(page.locator('.product')).toHaveCount(5);
```

### Text assertions

```javascript
await expect(page.locator('#message'))
    .toHaveText('Login Successful');

await expect(page.locator('#message'))
    .toContainText('Successful');
```

### Input and state assertions

```javascript
await expect(page.locator('#username')).toHaveValue('Arun');
await expect(page.locator('#remember')).toBeChecked();
await expect(page.locator('#login')).toBeEnabled();
await expect(page.locator('#login')).toBeDisabled();
await expect(page.locator('#username')).toBeEditable();
await expect(page.locator('#username')).toHaveAttribute('type', 'text');
await expect(page.locator('#login')).toHaveClass('login-button');
```

### Generic assertions for JavaScript values

```javascript
const marks = 90;
expect(marks).toBe(90);

const student = { name: 'Arun', marks: 90 };
expect(student).toEqual({ name: 'Arun', marks: 90 });

const students = ['Arun', 'Bala', 'Chitra'];
expect(students).toContain('Arun');
expect(students).toHaveLength(3);
expect(marks).toBeGreaterThan(80);
```


### Web-first assertions versus immediate checks

A web-first assertion retries until the condition passes or its timeout is reached:

```javascript
await expect(page.locator('#success')).toBeVisible();
```

A generic assertion checks a JavaScript value immediately:

```javascript
const result = 10;
expect(result).toBe(10);
```

Compare `isVisible()` with `toBeVisible()`:

```javascript
// Immediate Boolean result: true or false
const visible = await page.locator('#success').isVisible();
console.log(visible);

// Web-first assertion: retries until success or timeout
await expect(page.locator('#success')).toBeVisible();
```

| Feature | `isVisible()` | `toBeVisible()` |
|---|---|---|
| Type | Locator method | Assertion |
| Returns | Boolean | Assertion result |
| Auto-retry | No | Yes |
| Fails test automatically | No | Yes |
| Purpose | Immediate visibility check | Verify visibility |

Use `toBeVisible()` for test validations. Use `isVisible()` when an immediate Boolean result is needed for conditional logic.


## 5. Complete practical test case  

**Practice website:** https://testautomationpractice.blogspot.com/

**Exercise:** Verify the page title and interact with the Name textbox.

Save as `tests/day1-practice.spec.js`:

```javascript
const { test, expect } = require('@playwright/test');

test('Verify page and enter student name', async ({ page }) => {

    // Step 1: Navigate to the application
    await page.goto(
        'https://testautomationpractice.blogspot.com/'
    );

    // Step 2: Verify the page title
    await expect(page).toHaveTitle(
        /Automation Testing Practice/
    );

    // Step 3: Identify the Name textbox
    const name = page.locator('#name');

    // Step 4: Verify textbox visibility
    await expect(name).toBeVisible();

    // Step 5: Enter the student name
    await name.fill('Arun');

    // Step 6: Verify the entered value
    await expect(name).toHaveValue('Arun');

    // Step 7: Verify textbox is enabled
    await expect(name).toBeEnabled();

});
```

### Explain each step

1. `require('@playwright/test')` imports the test runner and assertion library.
2. `test()` defines a test case.
3. `async ({ page })` receives the built-in page fixture.
4. `page.goto()` navigates to the application.
5. `toHaveTitle()` verifies the browser page title.
6. `page.locator('#name')` identifies the Name textbox by ID.
7. `toBeVisible()` verifies that the textbox is visible.
8. `fill('Arun')` enters the text.
9. `toHaveValue('Arun')` verifies the textbox value.
10. `toBeEnabled()` verifies that the textbox is enabled.

Ask students to predict the result of each step before running the test. Then change the entered name and expected value.


## 6. Five-minute recap 

Ask students:

1. What is the difference between `locator()` and `getByRole()`?
2. What is the difference between `fill()` and `type()`?
3. Why does `nth(2)` identify the third element?
4. What is the difference between `toBeVisible()` and `isVisible()`?
5. Why do we prefer web-first assertions?

### Assertion quick reference

| Category | Assertions |
|---|---|
| Page | `toHaveTitle()`, `toHaveURL()` |
| Locator | `toBeVisible()`, `toBeHidden()`, `toBeAttached()`, `toHaveCount()` |
| Text | `toHaveText()`, `toContainText()` |
| Input | `toHaveValue()`, `toBeEmpty()`, `toBeChecked()` |
| State | `toBeEnabled()`, `toBeDisabled()`, `toBeEditable()` |
| Attribute | `toHaveAttribute()`, `toHaveClass()` |
| Generic | `toBe()`, `toEqual()`, `toContain()`, `toBeTruthy()` |
| Numeric | `toBeGreaterThan()`, `toBeLessThan()` |
| Array/string | `toHaveLength()` |


## 7. Student practical assignment  

Use https://testautomationpractice.blogspot.com/ and complete these tasks independently:

1. Enter a student name — `fill()`.
2. Enter an email address — `fill()`.
3. Enter a phone number — `fill()`.
4. Select a gender option — `check()`.
5. Select multiple days — checkbox handling.
6. Verify selected checkboxes — `toBeChecked()`.
7. Count all links — `count()`.
8. Verify the page heading — `getByRole()`.
9. Verify the entered name — `toHaveValue()`.
10. Verify the Submit button is visible and enabled — `toBeVisible()` and `toBeEnabled()`.

### Assignment rules
- Write your own locators; do not simply copy the teacher's code.
- Use meaningful variable names.
- Add comments explaining important steps.
- Use assertions to validate expected results.
- Execute the test and investigate failures.


## 8. Day 1 completion checklist

- [ ] Understand what a locator is.
- [ ] Use `getByRole()`.
- [ ] Use `getByText()`.
- [ ] Use `getByLabel()`.
- [ ] Use `getByPlaceholder()`.
- [ ] Use `getByAltText()`.
- [ ] Use `getByTestId()`.
- [ ] Use CSS and XPath locators.
- [ ] Perform basic locator actions.
- [ ] Handle multiple matching elements.
- [ ] Understand Playwright assertions.
- [ ] Understand web-first assertion retry behavior.
- [ ] Write and execute a complete test case.
- [ ] Complete the practical assignment.
